# 11. Sampling: Euler, CFG, schedules и NFE

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 1](https://diffusion.csail.mit.edu/2026/docs/20260120_Lecture_01.pdf) · [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf) · [Lecture 3](https://diffusion.csail.mit.edu/2026/docs/20260123_Lecture_03.pdf).
<!-- study-navigation:end -->

**Результат:** измерить ошибки solver и фактическое число calls в настоящем native denoise_loop

**Перед началом:** K6 09–10; Lecture 1 и Lecture 3. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/07_sampling.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §5](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/algo/flow_matching.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/flow_matching.py)
- [`core/algo/guidance.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/guidance.py)
- [`core/algo/denoise_loop.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/denoise_loop.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','main'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '11_sampling_cfg_and_nfe'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Solver интегрирует поле, но не обучает его

$$
x_{t+\Delta t}=x_t+\Delta t\,u_\theta(x_t,t,c),\qquad \Delta t<0.
$$

Для u(x,t)=x, x(1)=1, точное x(0)=exp(-1). Это удобный контроль: известны поле и ответ,
поэтому вся ошибка здесь — numerical integration. Для trained модели появится ещё approximation error.
Scheduler shift меняет узлы интегрирования; он не создаёт новую обученную модель.


In [ ]:
schedule=native('core.algo.flow_matching').flow_match_timesteps
rows=[]
for shift in [1.,5.]:
    for steps in [4,8,16,32,64]:
        ts=schedule(steps,shift,'cpu').double();x=torch.tensor(1.,dtype=torch.float64)
        for dt in torch.diff(ts):x=x+dt*x
        rows.append({'shift':shift,'NFE':steps,'error':abs(x.item()-math.exp(-1))})
fig,ax=plt.subplots(figsize=(7,3))
for shift in [1.,5.]:
    sel=[r for r in rows if r['shift']==shift]
    ax.loglog([r['NFE'] for r in sel],[r['error'] for r in sel],'o-',label=f'shift={shift}')
ax.legend();ax.set_xlabel('Field evaluations');ax.set_ylabel('Known-ODE endpoint error')
fig.tight_layout();fig.savefig(figure_path(LAB,'euler_error'));plt.show();metrics['known_ode_errors']=rows


## 2. Guidance — комбинация двух predictions

$$
u_{\mathrm{cfg}}=u_{\varnothing}+w(u_c-u_{\varnothing}).
$$

w=0 возвращает unconditional, w=1 — conditional. Native non-distilled Euler loop при w=1
пропускает второй forward. При w!=1 выполняет оба. Мы проверим это **настоящим loop**,
подставив явно аналитический constant predictor: он не умеет генерировать AV.
Для Lite Distill используем π-Flow из предыдущей лабораторной; CFG sweep ниже относится к обычному FM loop.


In [ ]:
Bundle=native('core.types').LatentBundle
class ConstantFieldDouble(nn.Module):
    visual_cond=True
    def __init__(self):super().__init__();self.calls=0
    def forward(self,**kw):
        self.calls+=1
        level=kw['text_embed'].mean()
        return torch.ones_like(kw['x_video'][...,:4])*level,torch.ones_like(kw['x_audio'])*level
base_v=torch.zeros(1,2,4,4,4);base_a=torch.zeros(1,6,4)
cu_v=torch.tensor([0,2],dtype=torch.int32);cu_a=torch.tensor([0,6],dtype=torch.int32)
cond={'text_embeds':torch.ones(1,3,12),'pooled_embed':torch.zeros(1,8)}
null={'text_embeds':torch.ones(1,3,12)*.25,'pooled_embed':torch.zeros(1,8)}
results=[]
for w in [0.,1.,3.]:
    field=ConstantFieldDouble();bundle=Bundle(base_v.clone(),base_a.clone(),cu_v,cu_a)
    out=native('core.algo.denoise_loop').denoise_loop(bundle,field,cond,null,None,None,
        torch.empty(3,1),torch.empty(3,1),num_steps=4,guidance_weight=w,scheduler_scale=5.)
    expected=-(.25+w*(1-.25))
    error=max((out.video-expected).abs().max().item(),(out.audio-expected).abs().max().item())
    assert error<1e-5 and field.calls==(4 if w==1 else 8)
    results.append({'guidance':w,'model_NFE':field.calls,'endpoint_error':error})
metrics['native_euler_cfg_checks']=results;print(results)


## 3. Что замерять в промышленном опыте

Указывайте checkpoint, sampler implementation, steps, model NFE, shift, dtype и cache.
Отдельно измеряйте weight loading, text encoding, denoising, VAE decode, vocoder и mux.
Warmup/compile не смешивайте с steady-state generation; median нескольких прогонов лучше одного числа.
Наш constant-field контроль не оценивает speedup большого checkpoint и не подбирает оптимальный CFG.


## Самостоятельное исследование

1. При одной trained модели меняйте только solver steps; затем при одном solver сравните training checkpoints.
2. Добавьте отдельный счетчик policy.pi evaluations в π-Flow и сравните с model NFE.
3. На constant field проверьте, почему Euler точен для любого partition времени, а на u=x — уже нет.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы различаете solver error, model error, guidance strength и число transformer calls.
Фраза «10 steps» без названия checkpoint/sampler не определяет вычислительный эксперимент.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 10. Lite Distill: grid-head, DX policy и π-Flow](10_piflow_distillation.ipynb) · [12. Latents, I2AV references, decode и output contracts →](12_latents_conditioning_and_output.ipynb)
